# WES : mosdepth fore PRE_BAM

In [4]:
import pandas as pd
import numpy as np
import os
import glob

hg = "hg38"
BAM_DIR = f"/data/project/Meningioma/02.Align/{hg}"

df_acc = pd.DataFrame(columns=["length", "bases", "mean", "min", "max", "DATE", "TISSUE"])

for TISSUE in ["Blood", "Tumor", "Dura", "Ventricle", "Cortex"]:
    file_list = [file for file in glob.glob(f"{BAM_DIR}/{TISSUE}/07.mosdepth/*.mosdepth.summary.txt") if "WGS" not in file]

    for file in file_list:
        DATE = os.path.basename(file).split(f"_{TISSUE}")[0]

        df_individual = pd.read_csv(file, sep="\t")

        # 마지막 row의 1열부터 가져오고 DATE, TISSUE 추가
        t = df_individual.iloc[-1][1:]
        t["DATE"] = DATE
        t["TISSUE"] = TISSUE

        df_acc = pd.concat([df_acc, pd.DataFrame([t])], ignore_index=True)
                
df_acc.sort_values(by=["TISSUE", "DATE"], ascending=True, inplace=True)  # 오름차순으로 정렬
df_acc.reset_index(drop=True, inplace=True)  # 인덱스 초기화

df_acc.to_csv ( "/data/project/Meningioma/script/02.Align&DOC/mosdepth.result.csv", index = False, sep = "\t")
pd.set_option('display.max_rows', 500)
df_acc

,length,bases,mean,min,max,DATE,TISSUE
0,50446305,6261721067,124.13,0,2093,190426,Blood
1,50446305,6816639706,135.13,0,2173,190524,Blood
2,50446305,7296865098,144.65,0,5362,220930_2,Blood
3,50446305,6084734471,120.62,0,4445,221026,Blood
4,50446305,6441855244,127.70,0,852,221102,Blood
5,50446305,5639973375,111.80,0,1779,221202,Blood
6,50446305,8601369677,170.51,0,4855,230127,Blood
7,50446305,4292154282,85.08,0,1313,230303_1,Blood
8,50446305,8207039116,162.69,0,3420,230323_11,Blood
9,50446305,8206997469,162.69,0,3436,230323_2,Blood


In [3]:
df_acc.groupby("TISSUE")["mean"].agg(["count", "mean", "std"]).reset_index().round(1)

,TISSUE,count,mean,std
0,Blood,30,127.3,22.2
1,Dura,26,341.3,76.4
2,Tumor,53,122.4,14.1


# WES : Depth of Coverage

In [10]:
import pandas as pd
import numpy as np
import os
import glob

hg="hg38"
BAM_DIR="/data/project/Meningioma/02.Align/" + hg

df = pd.DataFrame (columns = list (pd.read_csv ( "/data/project/Meningioma/02.Align/hg38/Dura/06.DepthOfCoverage/221026_Dura.depth.result.sample_summary" , sep = "," ).columns  ) + ["Target_depth", "Cost" ]  )
df = df.astype (float)
df["DATE"] = ""; df["TISSUE"] = ""
df = df.astype ({ 'sample_id':'str', "DATE": "str", "TISSUE" : "str"   })


for TISSUE in ["Blood", "Tumor", "Dura", "Ventricle", "Cortex"]:
    file_list = [file for file in glob.glob(BAM_DIR + "/" + TISSUE + "/06.DepthOfCoverage/*.depth.result.sample_summary") if "WGS" not in file]

    for file in file_list:
        DATE = os.path.basename(file).split( "_" + TISSUE )[0]

        df_individual = pd.read_csv ( file, sep = "," )

        if TISSUE in ["Dura", "Falx"]:
            target_depth = 900
            cost = 159.5
        elif TISSUE in [ "Tumor", "Ventricle", "Cortex", "Blood"]:
            target_depth = 300
            cost = 55
        


        t = pd.Series ( list(df_individual.iloc[0]) +  [target_depth, cost, DATE, TISSUE]  )
        t.index = df.columns
        
    
        df = pd.concat([df, pd.DataFrame([t])], ignore_index=True)
        


df.rename(columns = {"granular_median":"median" }, inplace = True)
df = df.astype ( {"median" : "int", "Target_depth" : "int"})


df [["sample_id", "DATE", "TISSUE", "Target_depth", "median", "Cost" ]].to_csv ( "/data/project/Meningioma/script/02.Align&DOC/depthofcoverage.result.csv", index = False, sep = "\t")
df [["sample_id", "DATE", "TISSUE", "Target_depth", "median", "mean" ]]


,sample_id,DATE,TISSUE,Target_depth,median,mean
0,230822_Blood,230822,Blood,300,108,106.25
1,241127_Blood,241127,Blood,300,129,128.25
2,221026_Blood,221026,Blood,300,123,141.26
3,221102_Blood,221102,Blood,300,116,112.42
4,230920_Blood,230920,Blood,300,111,109.98
5,240911_Blood,240911,Blood,300,115,113.93
6,240412_3_Blood,240412_3,Blood,300,97,96.38
7,240325_Blood,240325,Blood,300,144,140.28
8,240320_Blood,240320,Blood,300,138,137.02
9,230127_Blood,230127,Blood,300,159,182.83


In [18]:
df.groupby("TISSUE")["mean"].agg(["mean", "std"]).reset_index().round(1)

,TISSUE,mean,std
0,Blood,131.3,27.9
1,Cortex,252.6,NaN
2,Dura,310.9,59.2
3,Tumor,132.0,24.5
4,Ventricle,305.5,NaN


# AS; Read count 직접 계산

### Multiplex PCR 

In [43]:
multiplex_df = pd.read_csv("/data/project/Meningioma/51.Amplicon/08.multiplex_250227/07.pysam/multiplex_count_df.tsv", sep="\t",index_col = 0) 
multiplex_df_acc = pd.DataFrame ( columns = ["position", "read_depth"])
for col_i, col in enumerate ( multiplex_df.columns ):
    for row_i, row in enumerate ( multiplex_df.index ):
        ref_alt = multiplex_df.iloc [ row_i, col_i ]
        alt = int(ref_alt.split("/")[1])
        if alt != 0:
            multiplex_df_acc = pd.concat( [ multiplex_df_acc,  pd.DataFrame([[col, alt]], columns=["position", "read_depth"] ) ], ignore_index=True)


multiplex_df_acc.groupby("position")["read_depth"].agg(["mean", "std"]).reset_index().round(1)
multiplex_df_acc["read_depth"].agg(["mean", "std"]).round(1)

mean    1077216.0
std     1313202.8
Name: read_depth, dtype: float64

In [52]:
single_df_acc = pd.DataFrame ( columns = ["position",  "Sample_ID", "read_depth"])
file_list = [file for file in glob.glob( "/data/project/Meningioma/51.Amplicon/01.single/07.pysam/*count_df.tsv" ) ]

for file in file_list:
    POSITION = os.path.basename(file).split( "_count_df.tsv" )[0]
    print ( POSITION)
    
    single_df = pd.read_csv( file, sep="\t",index_col = 0) 
    for col_i, col in enumerate ( single_df.columns ):
        for row_i, row in enumerate ( single_df.index ):
            ref_alt = single_df.iloc [ row_i, col_i ]
            alt = int(ref_alt.split("/")[1])
            if alt != 0:
                single_df_acc = pd.concat( [ single_df_acc,  pd.DataFrame([[col, row, alt]], columns=["position", "Sample_ID", "read_depth"] ) ], ignore_index=True)
single_df_acc
single_df_acc["read_depth"].agg(["mean", "std"]).round(1)

chr22_29639168
chr22_29655663
chr22_29636758
chr16_2175609
chr14_104780214
chr16_2176145
chr9_107487067
chr22_29673365
chr22_29661335
chr22_29604113
chr16_2175939
chr22_29604050


mean    9886604.3
std     4186845.0
Name: read_depth, dtype: float64

---

### 전체 data의 DOC를 확인해보기

In [5]:
import pandas as pd
import numpy as np
import os

hg="hg38"
DATA_PATH="/data/project/Meningioma/02.Align/" + hg
datenames = sorted ( [line.rstrip('\n') for line in open('/data/project/Meningioma/script/sample_name.txt', 'r')]  )  # ['220930', '220930', '220930']



df = pd.DataFrame (columns = list (pd.read_csv ( "/data/project/Meningioma/02.Align/hg38/Dura/06.DepthOfCoverage/221026_Dura.depth.result.sample_summary" , sep = "," ).columns  ) + ["Target_depth", "Cost", "Corp"]  )
df = df.astype (float)
df = df.astype ({'sample_id':'str' , 'Corp':'str'  })


for tissue in ["Blood", "Tumor", "Dura", "Ventricle", "Cortex"]:
    for date in datenames:
        samplename = "{}_{}".format(date, tissue)
        INPUT_DOC =  DATA_PATH + "/" + tissue + "/06.DepthOfCoverage/" + samplename + ".depth.result.sample_summary"

        if os.path.exists (INPUT_DOC) == True:
            df_individual = pd.read_csv (INPUT_DOC, sep = "," )


            if date in ["220930", "221026", "230127", "230323", "230419"]:
                corp = "Theragen"
                if tissue == "Dura":
                    target_depth = 1000 if date == "221026" else 500
                    cost = 159.5 if date == "221026" else 107.25
                elif tissue in ["Ventricle", "Cortex"]:
                    target_depth = 500
                    cost = 107.25
                elif tissue in ["Blood", "Tumor"]:
                    target_depth = 300
                    cost = 88

            else:
                corp = "Macrogen"
                if tissue == "Dura":
                    target_depth = 500
                    cost = 99
                elif tissue in ["Ventricle", "Cortex"]:
                    target_depth = 500
                    cost = 99
                elif tissue in ["Blood", "Tumor"]:
                    target_depth = 300
                    cost = 49.5
            
            t = pd.Series ( list(df_individual.iloc[0]) +  [target_depth] + [cost] + [corp] )
            t.index = df.columns
            
        
            df = df.append( pd.Series(t) , ignore_index = True)    #개별 sample row만 합쳐준다
            


df.rename(columns = {"granular_median":"median" }, inplace = True)
df = df.astype ( {"median" : "int", "Target_depth" : "int"})


df [["sample_id", "Target_depth", "median", "Cost", "Corp"]].to_csv ( "/data/project/Meningioma/script/02.Align&DOC/depthofcoverage.result.csv", index = False, sep = "\t")
df [["sample_id", "Target_depth", "median", "mean", "Corp"]]


,sample_id,Target_depth,median,mean,Corp
0,220930_Blood,300,147,169.38,Theragen
1,221026_Blood,300,123,141.26,Theragen
2,221102_Blood,300,116,112.42,Macrogen
3,230127_Blood,300,159,182.83,Theragen
4,230323_Blood,300,161,183.33,Theragen
5,230405_Blood,300,110,106.03,Macrogen
6,230419_Blood,300,148,168.68,Theragen
7,220930_Tumor,300,125,150.31,Theragen
8,221026_Tumor,300,132,155.48,Theragen
9,221102_Tumor,300,108,109.44,Macrogen


In [4]:
df

,sample_id,total,mean,granular_third_quartile,median,granular_first_quartile,%_bases_above_15,Target_depth,Cost,Corp
0,220930_Blood,8.539370e+09,169.38,226.0,147,89.0,99.1,300,88.00,Theragen
1,221026_Blood,7.121894e+09,141.26,189.0,123,74.0,98.8,300,88.00,Theragen
2,221102_Blood,5.667659e+09,112.42,144.0,116,83.0,94.9,300,49.50,Macrogen
3,230127_Blood,9.217555e+09,182.83,244.0,159,97.0,99.2,300,88.00,Theragen
4,230323_Blood,9.242876e+09,183.33,245.0,161,99.0,99.1,300,88.00,Theragen
5,230405_Blood,5.345851e+09,106.03,136.0,110,78.0,94.8,300,49.50,Macrogen
6,230419_Blood,8.504073e+09,168.68,225.0,148,90.0,99.0,300,88.00,Theragen
7,220930_Tumor,7.578200e+09,150.31,201.0,125,73.0,98.5,300,88.00,Theragen
8,221026_Tumor,7.838875e+09,155.48,208.0,132,79.0,98.8,300,88.00,Theragen
9,221102_Tumor,5.517502e+09,109.44,141.0,108,75.0,94.9,300,49.50,Macrogen


# 연습장

In [3]:
maf_colnames = ['Chromosome', 'Start_Position', 'End_Position',
                'Tumor_Sample_Barcode', "ID", 'Hugo_Symbol', 'Reference_Allele',
                'Tumor_Seq_Allele2',  "Alt", "Depth", "BIOTYPE", 'Variant_Classification', "Impact",
                'tx', 'exon', "rsID", 'txChange', 'aaChange', 'Variant_Type', 'sample_id',
                'hgnc_symbol', 'Entrez', 'ens_id', 'Entrez_Gene_Id',
                "clinvar_MedGen_id", "clinvar_OMIM_id", "clinvar_Orphanet_id", "clinvar_clnsig", "clinvar_hgvs", "clinvar_id", "clinvar_review", "clinvar_trait", "clinvar_var_source",
                "CLIN_SIG", "SIFT", "PolyPhen",
                "SIFT_pred", "Polyphen2_HDIV_pred", "Polyphen2_HVAR_pred",
                "ClinPred_pred", "ClinPred_rankscore", "ClinPred_score", "DANN_rankscore", "DANN_score",
                "CADD", "GERP++_RS", "FATHMM_pred", "FATHMM_score", 'MetaSVM_pred',  "MetaSVM_rankscore", "MetaSVM_score", "MutationTaster_pred", 'MutationAssessor_pred',
                "SpliceAI_pred_DS_AG", "SpliceAI_pred_DS_AL", "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL", "SpliceAI_score",
                'MOTIF_NAME', 'MOTIF_POS', "MOTIF_SCORE_CHANGE", "TRANSCRIPTION_FACTORS", 'FunMotifs', "Nearest_gene",
                "KRG", "K1", "gnomAD", "dbSNP"]
noncoding_colnames = ["ChIP", "DNase", "PWM", "Footprint", "QTL", "PWM_matched", "Footprint_matched", "ranking_probability", "Non_Coding_Score", "Non_Coding_Groups", "Coding_Score", "Coding_Group", "EA_enhancer", "GH_promoter", "GH_enhancer", "RefSeq_promoter", "GH_promoter_enhancer", "ENSEMBL_promoter", "mTL_miRNA",
                      "greendb_id", "greendb_stdtype", "greendb_dbsource", "greendb_genes", "green_constraint", "greendb_level", "ANN"]

len(noncoding_colnames)

26